# Artificial and Computational Intelligence Assignment 1

## Problem Solving by AI Relief Supply Agent

## Team Members
List of all team members along with their BITS ID and contribution percentage:

| BITS ID       | Name                        | Contribution |
|--------------|-----------------------------|--------------|
| 2024AA05181  | Jagadi Shruti Chanabasappa  | 100%         |
| 2024AA05182  | Prathyusha Devi K           | 100%         |
| 2024AA05183  | A. Rohit Sai                | 100%         |
| 2024AA05184  | Sarwajit Kumar Mishra       | 100%         |
| 2024AA05185  | G V Ramya Sai               | 100%         |

## 1. PEAS (Performance, Environment, Actuators, Sensors)
The PEAS description for the Relief Supply Agent is summarized below:

| Aspect          | Description |
|----------------|-------------|
| **Performance Measure** | Minimize cost, optimize battery life, and ensure uninterrupted delivery. |
| **Environment** | Static, discrete, deterministic map of Ukraine along with path costs info. |
| **Actuators** | Movement across safe paths, delivery of supplies. |
| **Sensors** | Battery status, real-time path cost feedback. |

## Problem Statement
The goal is to develop an intelligent agent that delivers relief supplies to various cities in Ukraine while minimizing transportation costs. The agent must visit all cities and return to the starting point efficiently.

## Agent Data Structure Definition as per the Problem

In [49]:
# Graph Representation
graph = {
    "Kiev": {"Kharkiv": 10, "Odessa": 5, "Mikolaive": 15, "Lviv": 17},
    "Kharkiv": {"Kiev": 10, "Odessa": 7},
    "Odessa": {"Kharkiv": 7, "Kiev": 5, "Dnipro": 25},
    "Dnipro": {"Odessa": 25, "Kherson": 11, "Lviv": 4},
    "Lviv": {"Kiev": 17, "Mikolaive": 12, "Mariupol": 2, "Dnipro": 4, "Kherson": 10},
    "Mariupol": {"Lviv": 2},
    "Kherson": {"Dnipro": 11, "Lviv": 10},
    "Mikolaive": {"Kiev": 15, "Lviv": 12}
}

# Function to calculate complexities
def calculate_complexities(graph):
    vertices = len(graph)
    edges = sum(len(neighbors) for neighbors in graph.values()) // 2
    return vertices, edges


## Random Restart Hill Climbing Algorithm
Random Restart Hill Climbing is a local search algorithm that attempts to find an optimal solution by repeatedly restarting from different random initial states. It is particularly useful for avoiding local optima by exploring multiple starting points. The algorithm follows these steps:

1. Start from a randomly chosen node.
2. Expand the node and move to the neighbor with the lowest cost.
3. Repeat until no better moves are available.
4. Restart from a different random node and repeat the process.
5. Return the best solution found across all restarts.

In [50]:
import heapq

# Random Restart Hill Climbing Algorithm
def random_restart_hill_climbing(graph, start_node, restarts=10):
    best_path = None
    best_cost = float('inf')

    for _ in range(restarts):
        current_path = [start_node]
        current_cost = 0
        visited = {start_node}

        while len(visited) < len(graph):
            neighbors = [(neighbor, cost) for neighbor, cost in graph[current_path[-1]].items() if neighbor not in visited]
            if not neighbors:
                break
            next_city, travel_cost = min(neighbors, key=lambda x: x[1])
            current_path.append(next_city)
            current_cost += travel_cost
            visited.add(next_city)

        if len(visited) == len(graph) and start_node in graph[current_path[-1]]:
            current_cost += graph[current_path[-1]][start_node]
            current_path.append(start_node)

        if current_cost < best_cost:
            best_path, best_cost = current_path, current_cost

    return best_path, best_cost


## Heuristic and DFS Function
For heuristic design, consider all the possible paths between any arbitrary node `n` to the goal node (starting point). The average of the total transmission cost across all these paths is used as the heuristic value `h(n)`.

In [51]:
# Function to compute heuristic using DFS search
def dfs(node, goal_node, visited, graph, curr_path_cost, path_cost_list):
    if node == goal_node:
        path_cost_list.append(curr_path_cost)
        return
    visited[node] = True
    for neighbor, cost in graph[node].items():
        if not visited[neighbor]:
            dfs(neighbor, goal_node, visited, graph, curr_path_cost + cost, path_cost_list)
    visited[node] = False

# Compute Heuristic Value
def get_heuristic_value(start_node, goal_node):
    visited = {city: False for city in graph}
    path_cost_list = []
    dfs(start_node, goal_node, visited, graph, 0, path_cost_list)
    return int(round(sum(path_cost_list) / len(path_cost_list))) if path_cost_list else float('inf')


## A* Algorithm
A* is an informed search algorithm that finds the shortest path from the start node to the goal node by using a heuristic function to estimate the cost of reaching the goal. The algorithm follows these steps:

1. Initialize an open list and add the start node.
2. Expand the node with the lowest estimated total cost (f = g + h).
3. Update neighboring nodes and track the lowest-cost paths.
4. Continue until the goal node is reached.
5. Return the optimal path and total cost.

In [52]:
# A* Algorithm Implementation
def a_star_algorithm(goal_node, heuristic, vertices, edges):
    open_list = []
    closed_list = []
    heapq.heappush(open_list, (0 + heuristic[goal_node], goal_node, 0, heuristic[goal_node], [goal_node]))

    while open_list:
        f, curr_node, g, h, curr_path = heapq.heappop(open_list)
        if set(curr_path) == set(graph.keys()) and curr_node == goal_node:
            print("A* algorithm:")
            print(f"Start Node: {goal_node}")
            print("Final Path:", curr_path)
            print("Final Cost:", f)
            return curr_path
        closed_list.append(curr_node)
        for neighbor, cost in graph[curr_node].items():
            new_path = curr_path + [neighbor]
            heapq.heappush(open_list, (g + cost + heuristic[neighbor], neighbor, g + cost, heuristic[neighbor], new_path))


## Main Execution Output Analysis

In [53]:
# Main Execution
def main():
    start_node = input("Enter start node: ").strip()
    goal_node = start_node
    vertices, edges = calculate_complexities(graph)
    heuristic = {node: get_heuristic_value(node, goal_node) for node in graph}
    a_star_algorithm(start_node, heuristic, vertices, edges)

    print("\nRandom Restart Hill Climbing:")
    path, cost = random_restart_hill_climbing(graph, start_node)
    print(f"Start Node: {start_node}")
    print(f"Optimal Path: {path}")
    print(f"Total Cost: {cost}")

# Execute
main()


A* algorithm:
Start Node: Dnipro
Final Path: ['Dnipro', 'Kherson', 'Lviv', 'Mariupol', 'Lviv', 'Mikolaive', 'Kiev', 'Kharkiv', 'Odessa', 'Dnipro']
Final Cost: 94

Random Restart Hill Climbing:
Start Node: Dnipro
Optimal Path: ['Dnipro', 'Lviv', 'Mariupol']
Total Cost: 6


## Space and Time Complexity Analysis
### Random Restart Hill Climbing Algorithm
- Time Complexity: O(10 × V²) where V is the number of vertices.
- Space Complexity: O(V).




In [54]:
# Time & Space Complexity of Random Restart Hill Climbing
vertices, edges = calculate_complexities(graph)
print(f"Time Complexity of Random Restart Hill Climbing: O(10 × {vertices}^2)")
print(f"Space Complexity of Random Restart Hill Climbing: O({vertices})")

Time Complexity of Random Restart Hill Climbing: O(10 × 8^2)
Space Complexity of Random Restart Hill Climbing: O(8)


### A* Algorithm
- Time Complexity: O(E log V) where E is the number of edges and V is the number of vertices.
- Space Complexity: O(E + V).

In [55]:
# Time & Space Complexity of A* Algorithm
print(f"Time Complexity of A* algorithm: O({edges} log {vertices})")
print(f"Space Complexity of A* algorithm: O({edges} + {vertices})")

Time Complexity of A* algorithm: O(11 log 8)
Space Complexity of A* algorithm: O(11 + 8)


## Conclusion
The AI-based Relief Supply Agent successfully determines the shortest path using Random Restart Hill Climbing and A* Algorithm. While A* provides an optimal solution based on heuristics, Random Restart Hill Climbing helps explore multiple local optima to improve results.